In [1]:
# Cell 1: imports and solver config
import os
import glob
import pandas as pd
import numpy as np
import cobra as cb
import cobra
import scipy.stats as st
import matplotlib.pyplot as plt
import gifba
import micom

In [ ]:
INVITRO_ORIGINAL = "/home/rseag/UVM/M3_Lab/giFBA/Examples/2_real_models/data/invitro_original.csv"  # raw_data/invitro_original.csv from scfa_predictions repo
DM38_MEDIA       = "/home/rseag/UVM/M3_Lab/giFBA/Examples/2_real_models/data/DM38.csv"              # media/DM38.csv from scfa_predictions repo
AGORA_DIR        = "/home/rseag/UVM/M3_Lab/Butyrate_Prediction/Metabolic_Models/AGORA2_Models/"     # your local AGORA2 folder


In [ ]:
# build low-richness relative abundance table + measured butyrate (panel b)
raw = pd.read_csv(INVITRO_ORIGINAL, index_col=1)

raw[raw.columns[11:37]] = raw[raw.columns[11:37]].fillna(0).astype(int)
raw["richness"] = raw[raw.columns[11:37]].sum(axis=1)
raw["Plate"] = raw["Plate"].astype(str).str.split(".").str[0].str.zfill(2)
raw["Column"] = raw["Column"].astype(str).str.split(".").str[0].str.zfill(2)
raw["Run"] = raw["Sequencing Run"].str[-3:]
raw["sample_id"] = "P" + raw["Plate"] + raw["Row"] + raw["Column"] + "_" + raw["Run"]
raw = raw[raw["Contamination?"] == "No"].set_index("sample_id")

frac_cols = [c for c in raw.columns if "Fraction" in c and c != "B.cereus Fraction"]
rel_abundance = raw[frac_cols].round(4).dropna(how="all")
rel_abundance.columns = [c.split(" ")[0] for c in rel_abundance.columns]  # two-letter codes only

assert rel_abundance.columns[rel_abundance.columns.duplicated()].empty, "duplicate columns found"

low = raw[raw["richness"] == 26]
rel_abundance_low = rel_abundance.loc[rel_abundance.index.intersection(low.index)]

measured = raw["Butyrate"] / raw["OD"]
measured = measured[(measured >= 0) & (measured <= 100)]
measured_low = measured.loc[measured.index.intersection(rel_abundance_low.index)]

print(rel_abundance_low.shape, "samples x codes")
rel_abundance_low.head()

(895, 30) samples x codes


,BA,CA,BT,BU,PC,AC,BH,CC,CG,ER,...,CS,PJ,FP,EH,EC,BC,HB,BO,DL,BL
sample_id,,,,,,,,,,,,,,,,,,,,,
P19A01_003,0.0,0.0018,0.0,0.0,0.0034,0.0003,0.0,0.0000,0.0,0.9925,...,0.0,0.0,0.0013,0.0,0.0,0.0,0.0,0.0,0.0000,0.0
P19C01_003,0.0,0.0000,0.0,0.0,0.0000,0.7357,0.0,0.0000,0.0,0.2632,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0003,0.0
P19E01_003,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.8989,0.0,0.1006,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0002,0.0
P19E01_003,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0,0.8989,0.0,0.1006,...,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0002,0.0
P19G01_003,0.0,0.0000,0.0,0.0,0.0000,0.0002,0.0,0.0002,0.0,0.5427,...,0.0,0.0,0.0002,0.0,0.0,0.0,0.0,0.0,0.0000,0.0


In [4]:
media_df = pd.read_csv(DM38_MEDIA, index_col=0)

# media = {
#     row["reaction"].replace("_m", "(e)"): -abs(row["flux"])
#     for _, row in media_df.iterrows()
# }

media = dict(zip(media_df["reaction"], -abs(media_df["flux"])))

# add in media from Supplement 4 (not in DM38.csv?)
media["EX_glc_D_m"]  = -10  # 24.9783520948511 mM added(powder)
media["EX_arab_L_m"] = -10  # 21.314860454273 mM added (powder)
media["EX_lac_m"]    = -10  # 28.3081705150977 mM Sodium Lactate added (40% syrup)
media["EX_malt_m"]   = -10  # 4.38212094653812 mM added (powder)


list(media.items())[:5]

[('EX_4abz_m', -0.0073),
 ('EX_ade_m', -6e-06),
 ('EX_ala_L_m', -0.53),
 ('EX_arg_L_m', -2.2),
 ('EX_asn_L_m', -0.26)]

In [5]:
# Cell 5: resolve one AGORA2 model file per two-letter code (species match, genus fallback)
code_to_genus = {
    'PC': 'Prevotella',      'PJ': 'Parabacteroides', 'BV': 'Bacteroides',     'BF': 'Bacteroides',
    'BO': 'Bacteroides',     'BT': 'Bacteroides',     'BC': 'Bacteroides',     'BY': 'Bacteroides',
    'BU': 'Bacteroides',     'DP': 'Desulfovibrio',   'BL': 'Bifidobacterium', 'BA': 'Bifidobacterium',
    'BP': 'Bifidobacterium', 'CA': 'Collinsella',     'EL': 'Eggerthella',     'FP': 'Faecalibacterium',
    'CH': 'Clostridium',     'AC': 'Anaerostipes',    'BH': 'Blautia',         'CG': 'Clostridium',
    'ER': 'Eubacterium',     'RI': 'Roseburia',       'CC': 'Coprococcus',     'DL': 'Dorea',            'DF': 'Dorea',
}

code_to_species_guess = {
    "AC": "Anaerostipes_caccae",               "DP": "Desulfovibrio_piger",          "CG": "Clostridium_asparagiforme",
    "EL": "Eggerthella_lenta",                 "DF": "Dorea_formicigenerans",        "CA": "Collinsella_aerofaciens",
    "RI": "Roseburia_intestinalis",            "CH": "Clostridium_hiranonis",        "ER": "Eubacterium_rectale",
    "CC": "Coprococcus_comes",                 "FP": "Faecalibacterium_prausnitzii", "PC": "Prevotella_copri",
    "BV": "Bacteroides_vulgatus",              "BO": "Bacteroides_ovatus",           "BT": "Bacteroides_thetaiotaomicron",
    "BU": "Bacteroides_uniformis",             "BH": "Blautia_hydrogenotrophica",    "DL": "Dorea_longicatena",
    "PJ": "Parabacteroides_johnsonii",         "BF": "Bacteroides_fragilis",         "BC": "Bacteroides_caccae",
    "BL": "Bifidobacterium_longum",            "BA": "Bifidobacterium_adolescentis",
    "BP": "Bifidobacterium_pseudocatenulatum", "BY": "Bacteroides_cellulosilyticus"
}

def pick_best(files):
    preferred = [f for f in files if ("DSM" in f or "ATCC" in f)]
    return preferred[0] if preferred else (files[0] if files else None)

code_to_model = {}
for code, genus in code_to_genus.items():
    species = code_to_species_guess.get(code)
    files = glob.glob(os.path.join(AGORA_DIR, f"{species}*.mat")) if species else []
    source = "species"
    if not files:
        files = glob.glob(os.path.join(AGORA_DIR, f"{genus}_*.mat"))
        source = "genus fallback"
    chosen = pick_best(files)
    code_to_model[code] = chosen
    print(f"{code}: {os.path.basename(chosen) if chosen else 'NOT FOUND'}  ({source}, {len(files)} candidates)")

missing = [c for c, f in code_to_model.items() if f is None]
print("\nMissing entirely:", missing)

PC: Prevotella_copri_CB7_DSM_18205.mat  (species, 1 candidates)
PJ: Parabacteroides_johnsonii_DSM_18315.mat  (species, 1 candidates)
BV: Bacteroides_vulgatus_ATCC_8482.mat  (species, 1 candidates)
BF: Bacteroides_fragilis_NCTC_9343.mat  (species, 1 candidates)
BO: Bacteroides_ovatus_ATCC_8483.mat  (species, 1 candidates)
BT: Bacteroides_thetaiotaomicron_VPI_5482.mat  (species, 1 candidates)
BC: Bacteroides_caccae_ATCC_43185.mat  (species, 1 candidates)
BY: Bacteroides_cellulosilyticus_DSM_14838.mat  (species, 1 candidates)
BU: Bacteroides_uniformis_ATCC_8492.mat  (species, 1 candidates)
DP: Desulfovibrio_piger_ATCC_29098.mat  (species, 1 candidates)
BL: Bifidobacterium_longum_infantis_ATCC_15697.mat  (species, 1 candidates)
BA: Bifidobacterium_adolescentis_ATCC_15703.mat  (species, 1 candidates)
BP: Bifidobacterium_pseudocatenulatum_DSM_20438.mat  (species, 1 candidates)
CA: Collinsella_aerofaciens_ATCC_25986.mat  (species, 1 candidates)
EL: Eggerthella_lenta_DSM_2243.mat  (species, 1 

In [6]:
# # test if models grow alone on DM38 medium
# row = rel_abundance_low.iloc[0]
# # input(f"{row.name}")
# present = row[row > 0]
# present = present[present.index.isin(code_to_model.keys())]
# present = present[[code_to_model[c] is not None for c in present.index]]
# codes_present = present.index.tolist()
# abund = (present / present.sum()).to_numpy()
# models = [cb.io.load_matlab_model(code_to_model[c]) for c in codes_present]


# for model in models:
#     for ex in model.exchanges:
#         ex.lower_bound = 0
#         ex.upper_bound = 1000
#     for ex, flux in media.items():
#         if ex.endswith("_m"):
#             ex_id = ex[:-2] + "(e)"
#         if ex_id in model.reactions:
#             model.reactions.get_by_id(ex_id).lower_bound = flux
    
#     solution = model.optimize()
#     print(f"{model.name}: growth={solution.objective_value:.4f}")

In [7]:
# # replace _m from end of media keys with (e) to match model exchange IDs - ignoring _m if in middle of string
# media_agora = {}
# for k, v in media.items():
#     if k.endswith("_m"):
#         new_key = k[:-2] + "(e)"
#     else:
#         new_key = k
#     media_agora[new_key] = v
#     print(f"{k} -> {new_key}: {v}")
# # media_agora = {k[:-2] + "(e)" if k.endswith("_m") else k: v for k, v in media.items()}
# media_agora["EX_o2(e)"] = 0
# media_agora["EX_nac(e)"] = -1000
# media_agora["EX_cgly(e)"] = media_agora["EX_cys_L(e)"] + media_agora["EX_gly_L(e)"]
# media_agora["EX_glycys(e)"] = media_agora["EX_cys_L(e)"] + media_agora["EX_gly_L(e)"]
# media_agora["EX_alaasp(e)"] = media_agora["EX_ala_L(e)"] + media_agora["EX_asp_L(e)"]
# community = gifba.gifbaObject(models, [media_agora, 0.1], # minimal media
#                               rel_abund=abund) # relative abundance
# media_flux, org_flux = community.run_gifba(iters=10, method="pfba")
# for ex, flux in media_agora.items():
#     if ex not in community.env_fluxes.columns:
#         print(f"{ex} not in community.env_fluxes.columns")

# # get what media added and not in dm38


In [8]:
# added_media = set(community.media.keys()) - set(media_agora.keys())
# for ex in added_media:
#     print(ex)

In [9]:
# for idx, model in enumerate(community.models):
#     if "EX_nac(e)" in model.exchanges:
#         ex = model.exchanges.get_by_id("EX_nac(e)")
#         print(ex.id, ex.name)
#         print(f"{idx}: {model.name} has EX_nac(e) exchange")
#         break

In [ ]:
# simulate growth of each of the models as defined by table  using MICOM and using AGORA2 models
# simulate growth of each community in the low-richness abundance table using MICOM
# and the AGORA2 models matched to the two-letter code mapping defined above

micom_growth = {}
micom_failed = {}
micom_media = {ex: abs(flux) for ex, flux in media.items()}

for sample_id, row in rel_abundance_low.iterrows():
    present = row[row > 0]
    present = present[present.index.isin(code_to_model.keys())]
    present = present[[code_to_model[c] is not None for c in present.index]]

    if present.empty:
        continue

    codes_present = present.index.tolist()
    abund = (present / present.sum()).to_numpy()

    # models = [cb.io.load_matlab_model(code_to_model[c]) for c in codes_present]

    try:
        taxonomy = pd.DataFrame({
            "id": codes_present,
            "abundance": abund,
            "file": [code_to_model[c] for c in codes_present]
        })

        community = micom.Community(
            taxonomy=taxonomy,
            id=sample_id,
            progress=False
        )
        

        community.medium = media
        community.solver = "gurobi"

        solution = community.cooperative_tradeoff(fraction=0.7, pfba=True)

        if solution is None:
            raise RuntimeError("Simulation infeasible; returned None solution.")
            
        # solution.members is a DataFrame containing "abundance", "growth_rate", etc.
        growth = solution.members["growth_rate"]
        micom_growth[sample_id] = growth

        break

    except Exception as e:
        micom_failed[sample_id] = str(e)
        print(f"MICOM failed for {sample_id}: {e}")

if micom_growth:
    micom_growth_df = pd.DataFrame(micom_growth).T
    print(f"MICOM growth predictions for {micom_growth_df.shape[0]} samples")
    micom_growth_df.head()
else:
    micom_growth_df = pd.DataFrame()
    print("No MICOM predictions were generated.")

if micom_failed:
    print("Failed sample IDs:", list(micom_failed.keys()))

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expressi

[08/31/26 11:01:40] WARNING  solver encountered an error infeasible_or_unbounded                    ]8;id=369733;file:///home/rseag/anaconda3/envs/M3/lib/python3.10/site-packages/micom/solution.py\solution.py]8;;\:]8;id=466303;file:///home/rseag/anaconda3/envs/M3/lib/python3.10/site-packages/micom/solution.py#206\206]8;;\

                    WARNING  solver encountered an error infeasible_or_unbounded                    ]8;id=973282;file:///home/rseag/anaconda3/envs/M3/lib/python3.10/site-packages/micom/solution.py\solution.py]8;;\:]8;id=477535;file:///home/rseag/anaconda3/envs/M3/lib/python3.10/site-packages/micom/solution.py#206\206]8;;\

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


MICOM failed for P19A01_003: could not get community growth rate.


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


In [10]:
for ex in community.exchanges:
    print(ex.id)

for ex, flux in media.items():
    if ex not in [e.id for e in community.exchanges]:
        print(f"{ex}: {flux}")

EX_12dhchol_m
EX_26dap_M_m
EX_2ddglcn_m
EX_3dhcdchol_m
EX_3dhchol_m
EX_7a_czp_m
EX_7dhcdchol_m
EX_7ocholate_m
EX_C02528_m
EX_HC02194_m
EX_ac_m
EX_acald_m
EX_acgam_m
EX_acnam_m
EX_adocbl_m
EX_ala_L_m
EX_alaasp_m
EX_alagln_m
EX_alaglu_m
EX_alagly_m
EX_alahis_m
EX_alaleu_m
EX_alathr_m
EX_amylopect900_m
EX_anzp_m
EX_arab_L_m
EX_arbt_m
EX_arg_L_m
EX_ca2_m
EX_cbl1_m
EX_cbl2_m
EX_cd2_m
EX_cgly_m
EX_chlphncl_m
EX_cholate_m
EX_cl_m
EX_co2_m
EX_cobalt2_m
EX_cu2_m
EX_czp_m
EX_dgchol_m
EX_etoh_m
EX_fe2_m
EX_fe3_m
EX_fol_m
EX_for_m
EX_fru_m
EX_gal_m
EX_galt_m
EX_gam_m
EX_gchola_m
EX_glc_D_m
EX_gln_L_m
EX_gly_m
EX_glyasn_m
EX_glyasp_m
EX_glyc_m
EX_glycys_m
EX_glygln_m
EX_glyglu_m
EX_glyleu_m
EX_glymet_m
EX_glyphe_m
EX_glypro_m
EX_glytyr_m
EX_h_m
EX_h2_m
EX_h2o_m
EX_hg2_m
EX_hxan_m
EX_ile_L_m
EX_inulin_m
EX_k_m
EX_lac_D_m
EX_lac_L_m
EX_lcts_m
EX_leu_L_m
EX_malt_m
EX_malttr_m
EX_man_m
EX_met_L_m
EX_metala_m
EX_mg2_m
EX_mn2_m
EX_na1_m
EX_nac_m
EX_nchlphncl_m
EX_nh4_m
EX_no3_m
EX_nzp_m
EX_o2_m
EX_orn_m


In [7]:
# Cell 6: run giFBA per low-richness sample, collect predicted butyrate (fixed filtering)
BUTYRATE_EX = "EX_but(e)"

_model_cache = {}
def load_code_model(code):
    if code not in _model_cache:
        _model_cache[code] = cb.io.load_matlab_model(code_to_model[code])
    return _model_cache[code]

predicted_butyrate = {}
skipped_unknown_codes = set()

for sample_id, row in rel_abundance_low.iterrows():
    present = row[row > 0]

    unknown = present.index.difference(code_to_model.keys())
    if len(unknown) > 0:
        skipped_unknown_codes.update(unknown)
    present = present[present.index.isin(code_to_model.keys())]

    present = present[[code_to_model[c] is not None for c in present.index]]
    if present.empty:
        continue

    codes_present = present.index.tolist()
    abund = (present / present.sum()).tolist()
    models = [load_code_model(c) for c in codes_present]

    community = gifba.gifbaObject(models, media, rel_abund=abund)
    media_flux, org_flux = community.run_gifba(iters=5, method="pfba", v=False)

    predicted_butyrate[sample_id] = org_flux[BUTYRATE_EX].sum() if BUTYRATE_EX in org_flux.columns else None

predicted_butyrate = pd.Series(predicted_butyrate, name="predicted")
print(f"{predicted_butyrate.notna().sum()} / {len(rel_abundance_low)} samples produced a prediction")
print("Unknown codes encountered (dropped):", skipped_unknown_codes)
predicted_butyrate.head()

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expressi

Read LP format model from file /tmp/tmpr3xshrjh.lp
Reading time = 0.00 seconds
: 832 rows, 1772 columns, 7534 nonzeros
Read LP format model from file /tmp/tmpct83lh5o.lp
Reading time = 0.01 seconds
: 1621 rows, 3724 columns, 12098 nonzeros
Read LP format model from file /tmp/tmp5rum1dlg.lp
Reading time = 0.00 seconds
: 1080 rows, 2348 columns, 9984 nonzeros
Read LP format model from file /tmp/tmpa02y6xy9.lp
Reading time = 0.00 seconds
: 1110 rows, 2554 columns, 10684 nonzeros
Read LP format model from file /tmp/tmpveirl0_k.lp
Reading time = 0.00 seconds
: 965 rows, 2070 columns, 8870 nonzeros
Read LP format model from file /tmp/tmpmcaviym8.lp
Reading time = 0.00 seconds
: 1128 rows, 2458 columns, 10716 nonzeros
Read LP format model from file /tmp/tmpjvydbcth.lp
Reading time = 0.01 seconds
: 1701 rows, 4020 columns, 14182 nonzeros
Relative abundances set to: [1.80054016e-03 3.40102031e-03 3.00090027e-04 9.92797839e-01
 2.00060018e-04 2.00060018e-04 1.30039012e-03]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


Read LP format model from file /tmp/tmpxwdc95ui.lp
Reading time = 0.00 seconds
: 1080 rows, 2348 columns, 9984 nonzeros
Read LP format model from file /tmp/tmphl7zgodx.lp
Reading time = 0.00 seconds
: 1110 rows, 2554 columns, 10684 nonzeros
Read LP format model from file /tmp/tmpopbeq15u.lp
Reading time = 0.00 seconds
: 965 rows, 2070 columns, 8870 nonzeros
Read LP format model from file /tmp/tmpkb5coqzy.lp
Reading time = 0.01 seconds
: 1691 rows, 3966 columns, 15066 nonzeros
Read LP format model from file /tmp/tmprjvyb3nd.lp
Reading time = 0.00 seconds
: 973 rows, 2078 columns, 8850 nonzeros
Relative abundances set to: [7.35994398e-01 2.63305322e-01 2.00080032e-04 2.00080032e-04
 3.00120048e-04]


...

Read LP format model from file /tmp/tmpcjbjupxd.lp
Reading time = 0.01 seconds
: 1983 rows, 4674 columns, 15424 nonzeros
Read LP format model from file /tmp/tmpdjpos6z8.lp
Reading time = 0.00 seconds
: 953 rows, 1980 columns, 8492 nonzeros
Read LP format model from file /tmp/tmpdr4lttl2.lp
Reading time = 0.01 seconds
: 1701 rows, 4020 columns, 14182 nonzeros
Read LP format model from file /tmp/tmpd70fxed0.lp
Reading time = 0.00 seconds
: 973 rows, 2078 columns, 8850 nonzeros
Relative abundances set to: [4.00160064e-04 3.20128051e-03 4.00160064e-04 4.60184074e-03
 4.00160064e-04 2.00080032e-04 2.00080032e-04 2.00080032e-04
 2.00080032e-04 9.90196078e-01]


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, p
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


Read LP format model from file /tmp/tmpwve3bk_8.lp
Reading time = 0.00 seconds
: 946 rows, 2106 columns, 8756 nonzeros
Read LP format model from file /tmp/tmptwyfcrco.lp
Reading time = 0.01 seconds
: 2041 rows, 4876 columns, 15788 nonzeros
Read LP format model from file /tmp/tmpkgmbp77e.lp
Reading time = 0.01 seconds
: 2204 rows, 5372 columns, 18058 nonzeros
Read LP format model from file /tmp/tmpcoulnmth.lp
Reading time = 0.01 seconds
: 1621 rows, 3724 columns, 12098 nonzeros
Read LP format model from file /tmp/tmpow669j9e.lp
Reading time = 0.01 seconds
: 984 rows, 2068 columns, 8872 nonzeros
Read LP format model from file /tmp/tmpatvszhg_.lp
Reading time = 0.01 seconds
: 1737 rows, 4048 columns, 14420 nonzeros
Read LP format model from file /tmp/tmpsn1ct8td.lp
Reading time = 0.00 seconds
: 1110 rows, 2554 columns, 10684 nonzeros
Read LP format model from file /tmp/tmpo5rnmvc2.lp
Reading time = 0.01 seconds
: 1987 rows, 4668 columns, 15444 nonzeros
Read LP format model from file /tmp/

: 

: 